<a href="https://colab.research.google.com/github/nov-cpu/8730-project/blob/API_Pulling/API_Pulling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 28-07-2026 (Initial Version)

# !pip install google-search-results pandas openpyxl

In [ ]:
# 28-07-2026 (Initial Version)

# import pandas as pd
# from serpapi import GoogleSearch
# import time
# import json

# # 1. Load the Data
# excel_path = '/content/alt_fuel_stations (Jul 26 2026).xlsx'

# # Read the Locations and Stations sheets
# df_locations = pd.read_excel(excel_path, sheet_name='Locations (Entity Table)')
# df_stations = pd.read_excel(excel_path, sheet_name='Stations (Parent Table)')

# # Clean up column names to avoid trailing whitespace issues
# df_locations.columns = df_locations.columns.str.strip()
# df_stations.columns = df_stations.columns.str.strip()

# # Merge the tables to associate the Station Name with its Address and Coordinates
# df_merged = pd.merge(
#     df_stations,
#     df_locations,
#     left_on='Location_ID (FK)',
#     right_on='Location_ID (PK)'
# )

# # 2. Setup SerpAPI Data Collection
# API_KEY = '4d175cb9ae871aa6a0c485c3512c0293d51453c4c26a4fe6f8dd1393ed92fb34'
# results_list = []

# # Note: We are using .head(5) to test the first 5 records and avoid burning API credits.
# # Remove .head(5) to run this across the entire dataset once you verify it works!
# for index, row in df_merged.head(5).iterrows():
#     station_name = row['Station_name']
#     street = row['street_Address']
#     city = row['City']
#     state = row['State']
#     lat = row['Latitude']
#     lon = row['Longitude']

#     # Construct a robust search query (e.g., "Ramada 1319 2nd St W Brooks AB")
#     search_query = f"{station_name} {street} {city} {state}"

#     params = {
#       "engine": "google_maps",
#       "q": search_query,
#       "ll": f"@{lat},{lon},15z", # Centers the map search around the exact lat/lon
#       "type": "search",
#       "api_key": API_KEY
#     }

#     try:
#         search = GoogleSearch(params)
#         results = search.get_dict()

#         place = None

#         # Check if an EXACT match exists (place_results)
#         if "place_results" in results:
#             place = results["place_results"]

#         # Fallback to checking if multiple local results exist
#         elif "local_results" in results and len(results["local_results"]) > 0:
#             place = results["local_results"][0]

#         if place:
#             # Extract the ratings, review count, and place information
#             extracted_data = {
#                 "Station_ID": row.get('Station_ID  (PK)'), # Using .get() is safer
#                 "Station_name": station_name,
#                 "Google_Place_ID": place.get("place_id"),
#                 "Google_Title": place.get("title"),
#                 "Rating": place.get("rating"),
#                 "Reviews_Count": place.get("reviews"),
#                 "Type": place.get("type"),
#                 "Address": place.get("address"),
#                 "Operating_Hours": place.get("operating_hours", {}).get("open_now")
#             }
#             results_list.append(extracted_data)
#             print(f"Successfully collected data for: {station_name}")
#         else:
#             print(f"No results found for: {search_query}")

#     except Exception as e:
#         print(f"Error fetching data for {search_query}: {e}")

#     # Respect API rate limits by pausing briefly between requests
#     time.sleep(1)

# # 3. Save the Extracted Data
# df_results = pd.DataFrame(results_list)

# # Export to JSON: Ideal for loading into MongoDB to handle unstructured/semi-structured data
# df_results.to_json('google_maps_station_ratings.json', orient='records', indent=4)

# # Export to CSV: Ideal for cleaning and loading into MySQL
# df_results.to_csv('google_maps_station_ratings.csv', index=False)

# print("\nData collection complete! JSON and CSV files have been saved.")

In [ ]:
# # 30-07-2026 (Version 2.0 - Individual Reviews Extraction)

# !pip install google-search-results pandas openpyxl numpy

# import pandas as pd
# from serpapi import GoogleSearch
# import time
# import numpy as np

# # ==========================================
# # 1. HAVERSINE FORMULA (DISTANCE CALCULATION)
# # ==========================================
# def calculate_distance(lat1, lon1, lat2, lon2):
#     R = 6371.0 # Earth radius in kilometers
#     lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

#     dlat = lat2 - lat1
#     dlon = lon2 - lon1

#     a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
#     c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

#     return R * c # Distance in kilometers

# # ==========================================
# # 2. CONFIGURATION VARIABLES
# # ==========================================
# API_KEY = '4d175cb9ae871aa6a0c485c3512c0293d51453c4c26a4fe6f8dd1393ed92fb34'
# EXCEL_PATH = '/content/alt_fuel_stations (Jul 26 2026).xlsx'

# # Coordinates for Superstore Dougall Avenue Dock, Windsor, ON
# USER_LAT = 42.2743
# USER_LON = -83.0039
# SEARCH_RADIUS_KM = 5.0
# API_LIMIT = 5 # Scrape top 5 closest stations to protect API limits

# # ==========================================
# # 3. LOAD AND FILTER DATA BY PROXIMITY
# # ==========================================
# df_locations = pd.read_excel(EXCEL_PATH, sheet_name='Locations (Entity Table)')
# df_stations = pd.read_excel(EXCEL_PATH, sheet_name='Stations (Parent Table)')

# df_locations.columns = df_locations.columns.str.strip()
# df_stations.columns = df_stations.columns.str.strip()

# df_merged = pd.merge(
#     df_stations,
#     df_locations,
#     left_on='Location_ID (FK)',
#     right_on='Location_ID (PK)'
# )

# df_merged['Distance_KM'] = calculate_distance(
#     USER_LAT, USER_LON,
#     df_merged['Latitude'], df_merged['Longitude']
# )

# df_filtered = df_merged[df_merged['Distance_KM'] <= SEARCH_RADIUS_KM].copy()

# total_nearby_count = len(df_filtered)
# print(f"\n--- Output 1: Proximity Count ---")
# print(f"There are {total_nearby_count} charging locations within {SEARCH_RADIUS_KM}km of the user.")
# print(f"---------------------------------\n")

# df_sorted = df_filtered.sort_values(by="Distance_KM")
# df_top_closest = df_sorted.head(API_LIMIT)

# print(f"--- Output 2: Scraping Details & Reviews for Top {API_LIMIT} Stations ---")

# # ==========================================
# # 4. SCRAPE MAPS & INDIVIDUAL REVIEWS
# # ==========================================
# results_list = []

# for index, row in df_top_closest.iterrows():
#     station_name = row['Station_name']
#     street = row['street_Address']
#     city = row['City']
#     state = row['State']
#     lat = row['Latitude']
#     lon = row['Longitude']
#     distance = round(row['Distance_KM'], 2)

#     search_query = f"EV Charging Station {street} {city} {state}"

#     params = {
#       "engine": "google_maps",
#       "q": search_query,
#       "ll": f"@{lat},{lon},18z",
#       "type": "search",
#       "api_key": API_KEY
#     }

#     try:
#         search = GoogleSearch(params)
#         results = search.get_dict()

#         place = None
#         if "place_results" in results:
#             place = results["place_results"]
#         elif "local_results" in results and len(results["local_results"]) > 0:
#             place = results["local_results"][0]

#         data_id = place.get("data_id") if place else None
#         user_reviews = []

#         # SECONDARY API CALL: Scrape reviews text if data_id is present
#         if data_id:
#             try:
#                 review_params = {
#                     "engine": "google_maps_reviews",
#                     "data_id": data_id,
#                     "api_key": API_KEY
#                 }
#                 review_search = GoogleSearch(review_params)
#                 review_results = review_search.get_dict()

#                 raw_reviews = review_results.get("reviews", [])
#                 for rev in raw_reviews:
#                     user_reviews.append({
#                         "Author": rev.get("user", {}).get("name", "Anonymous"),
#                         "Rating": rev.get("rating"),
#                         "Date": rev.get("date"),
#                         "Comment": rev.get("snippet", rev.get("text", "No text provided"))
#                     })
#                 print(f"  └─ Extracted {len(user_reviews)} individual reviews.")
#             except Exception as rev_err:
#                 print(f"  └─ Error fetching reviews: {rev_err}")

#         # Combine station details and extracted reviews
#         extracted_data = {
#             "Station_Name": station_name,
#             "Distance_km": distance,
#             "Address": place.get("address", f"{street}, {city}, {state}") if place else f"{street}, {city}, {state}",
#             "Rating": place.get("rating", "N/A") if place else "N/A",
#             "Reviews_Count": place.get("reviews", len(user_reviews)) if place else 0,
#             "Operating_Hours": place.get("operating_hours", {}).get("open_now", "Unknown") if place else "Unknown",
#             "Google_Maps_Link": f"https://www.google.com/maps?q={lat},{lon}",
#             "Detailed_User_Reviews": user_reviews # Nested JSON structure ideal for MongoDB
#         }

#         results_list.append(extracted_data)
#         print(f"Collected data: {station_name} ({distance} km away)")

#     except Exception as e:
#         print(f"Error fetching {search_query}: {e}")

#     time.sleep(1)

# # ==========================================
# # 5. GENERATE OUTPUT FILES (CSV, JSON, HTML)
# # ==========================================
# df_results = pd.DataFrame(results_list)

# if not df_results.empty:
#     # 1. Export JSON: Contains full nested user review objects (MongoDB ready)
#     df_results.to_json('nearby_station_ratings_with_reviews.json', orient='records', indent=4)

#     # 2. Export CSV: Flattens reviews into text summaries for MySQL tabular storage
#     df_csv = df_results.copy()
#     df_csv['Detailed_User_Reviews'] = df_csv['Detailed_User_Reviews'].apply(
#         lambda revs: " | ".join([f"[{r['Rating']}⭐] {r['Author']}: {r['Comment']}" for r in revs]) if revs else "No reviews"
#     )
#     df_csv.to_csv('nearby_station_ratings_with_reviews.csv', index=False)

#     # 3. Export Interactive HTML Table
#     df_html = df_results.copy()
#     df_html['Google_Maps_Link'] = df_html['Google_Maps_Link'].apply(
#         lambda x: f'<a href="{x}" target="_blank">View Pin on Maps</a>'
#     )
#     df_html['User_Reviews_Summary'] = df_html['Detailed_User_Reviews'].apply(
#         lambda revs: "<br>".join([f"• <b>{r['Author']} ({r['Rating']}⭐):</b> {r['Comment']}" for r in revs[:3]]) if revs else "<i>No text reviews available</i>"
#     )

#     # Drop nested column for clean display table
#     df_html_display = df_html[['Station_Name', 'Distance_km', 'Address', 'Rating', 'Reviews_Count', 'Operating_Hours', 'User_Reviews_Summary', 'Google_Maps_Link']]

#     html_output = f"""
#     <html>
#     <head>
#         <title>Closest {API_LIMIT} EV Charging Stations with Reviews</title>
#         <style>
#             body {{ font-family: Arial, sans-serif; margin: 20px; }}
#             h2 {{ color: #2c3e50; }}
#             table {{ border-collapse: collapse; width: 100%; margin-top: 20px; box-shadow: 0 2px 3px rgba(0,0,0,0.1); }}
#             th, td {{ padding: 12px; text-align: left; border-bottom: 1px solid #ddd; vertical-align: top; }}
#             th {{ background-color: #27ae60; color: white; }}
#             tr:hover {{ background-color: #f5f5f5; }}
#             a {{ color: #2980b9; text-decoration: none; font-weight: bold; }}
#             a:hover {{ text-decoration: underline; }}
#         </style>
#     </head>
#     <body>
#         <h2>⚡ Top {API_LIMIT} Closest EV Charging Stations & User Reviews</h2>
#         <p><strong>Base Location:</strong> Superstore Dougall Avenue Dock, Windsor, ON</p>
#         {{table}}
#     </body>
#     </html>
#     """

#     final_html = html_output.replace('{{table}}', df_html_display.to_html(escape=False, index=False))

#     with open("closest_stations_with_reviews.html", "w", encoding='utf-8') as f:
#         f.write(final_html)

#     print("\nSuccess! HTML, JSON (with nested reviews), and CSV files are ready in your Colab files panel.")
# else:
#     print("\nNo stations were found within the specified radius.")

In [ ]:
# # 31-07-2026 (Version 3.0 - Dynamic Location, Country Checks & Folium Mapping)

# !pip install google-search-results pandas openpyxl numpy geopy folium requests

# import pandas as pd
# from serpapi import GoogleSearch
# import time
# import numpy as np
# import requests
# import folium
# from geopy.geocoders import Nominatim

# # ==========================================
# # 1. HAVERSINE FORMULA (DISTANCE CALCULATION)
# # ==========================================
# def calculate_distance(lat1, lon1, lat2, lon2):
#     R = 6371.0 # Earth radius in kilometers
#     lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

#     dlat = lat2 - lat1
#     dlon = lon2 - lon1

#     a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
#     c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

#     return R * c

# # ==========================================
# # 2. CONFIGURATION VARIABLES
# # ==========================================
# API_KEY = '4d175cb9ae871aa6a0c485c3512c0293d51453c4c26a4fe6f8dd1393ed92fb34'
# EXCEL_PATH = '/content/alt_fuel_stations (Jul 26 2026).xlsx'
# SEARCH_RADIUS_KM = 5.0
# API_LIMIT = 5

# geolocator = Nominatim(user_agent="ev_charging_app_8730")

# # ==========================================
# # 3. GET DYNAMIC USER LOCATION (LIVE OR CUSTOM)
# # ==========================================
# user_lat, user_lon, user_country = None, None, None

# while True:
#     print("\n🌍 EV CHARGING STATION FINDER")
#     print("1. Use Current Live Location (IP-based)")
#     print("2. Enter a Custom Location")
#     choice = input("Select an option (1 or 2): ").strip()

#     if choice == '1':
#         print("\nFetching live location...")
#         try:
#             # Uses free IP-based location mapping
#             res = requests.get('http://ip-api.com/json/').json()
#             if res['status'] == 'success':
#                 user_lat, user_lon, user_country = res['lat'], res['lon'], res['country']
#         except Exception as e:
#             print("Could not fetch live location. Please use option 2.")
#             continue

#     elif choice == '2':
#         address = input("\nEnter a starting location (e.g., 'Windsor, ON, Canada'): ")
#         location = geolocator.geocode(address, exactly_one=True)
#         if location:
#             user_lat, user_lon = location.latitude, location.longitude
#             # Extract country string from the raw response
#             user_country = location.address.split(',')[-1].strip()
#         else:
#             print("Location not found. Please try again.")
#             continue
#     else:
#         print("Invalid choice. Please enter 1 or 2.")
#         continue

#     # --- CONDITION 2.3: Check if location is outside Canada ---
#     if 'Canada' not in user_country and 'CA' not in user_country:
#         print(f"\n❌ ERROR: Your location is detected as '{user_country}'.")
#         print("This application only supports locations within Canada.")
#         print("Please re-enter a valid Canadian location.\n")
#         continue # Loops back to the start
#     else:
#         print(f"\n✅ Location accepted! Searching near: {user_lat}, {user_lon} (Canada)")
#         break # Exits the loop and proceeds with the script

# # ==========================================
# # 4. LOAD AND FILTER DATA BY PROXIMITY
# # ==========================================
# df_locations = pd.read_excel(EXCEL_PATH, sheet_name='Locations (Entity Table)')
# df_stations = pd.read_excel(EXCEL_PATH, sheet_name='Stations (Parent Table)')

# df_locations.columns = df_locations.columns.str.strip()
# df_stations.columns = df_stations.columns.str.strip()

# df_merged = pd.merge(df_stations, df_locations, left_on='Location_ID (FK)', right_on='Location_ID (PK)')

# df_merged['Distance_KM'] = calculate_distance(user_lat, user_lon, df_merged['Latitude'], df_merged['Longitude'])
# df_filtered = df_merged[df_merged['Distance_KM'] <= SEARCH_RADIUS_KM].copy()

# total_nearby_count = len(df_filtered)

# # --- CONDITION 2.2: Handle NO nearby stations scenario ---
# if total_nearby_count == 0:
#     min_dist = df_merged['Distance_KM'].min()
#     print(f"\n⚠️ Output: Sorry, there is no charging point option closer (< {SEARCH_RADIUS_KM}kms) to your current location.")
#     print(f"However, if you travel {min_dist:.2f} kms, you can reach the closest available charging station!")

#     # We still isolate the 1 closest station so we can map it for the user
#     df_top_closest = df_merged.sort_values(by="Distance_KM").head(1)
#     scrape_limit = 1
# else:
#     # --- CONDITION 2.1: Handle normal scenario with nearby stations ---
#     print(f"\n--- Output 1: Proximity Count ---")
#     print(f"There are {total_nearby_count} charging locations within {SEARCH_RADIUS_KM}km of your location.")
#     print(f"---------------------------------\n")

#     df_sorted = df_filtered.sort_values(by="Distance_KM")
#     df_top_closest = df_sorted.head(API_LIMIT)
#     scrape_limit = API_LIMIT

# print(f"--- Output 2: Scraping Details & Reviews for Top {scrape_limit} Station(s) ---")

# # ==========================================
# # 5. SCRAPE MAPS & INDIVIDUAL REVIEWS
# # ==========================================
# results_list = []

# for index, row in df_top_closest.iterrows():
#     station_name = row['Station_name']
#     street = row['street_Address']
#     city = row['City']
#     state = row['State']
#     lat = row['Latitude']
#     lon = row['Longitude']
#     distance = round(row['Distance_KM'], 2)

#     search_query = f"EV Charging Station {street} {city} {state}"
#     params = {"engine": "google_maps", "q": search_query, "ll": f"@{lat},{lon},18z", "type": "search", "api_key": API_KEY}

#     try:
#         search = GoogleSearch(params)
#         results = search.get_dict()
#         place = results.get("place_results", results.get("local_results", [None])[0] if "local_results" in results else None)

#         data_id = place.get("data_id") if place else None
#         user_reviews = []

#         if data_id:
#             try:
#                 review_params = {"engine": "google_maps_reviews", "data_id": data_id, "api_key": API_KEY}
#                 review_search = GoogleSearch(review_params)
#                 raw_reviews = review_search.get_dict().get("reviews", [])
#                 for rev in raw_reviews:
#                     user_reviews.append({
#                         "Author": rev.get("user", {}).get("name", "Anonymous"),
#                         "Rating": rev.get("rating"),
#                         "Comment": rev.get("snippet", rev.get("text", "No text provided"))
#                     })
#                 print(f"  └─ Extracted {len(user_reviews)} reviews.")
#             except:
#                 pass

#         extracted_data = {
#             "Station_Name": station_name,
#             "Distance_km": distance,
#             "Latitude": lat,
#             "Longitude": lon,
#             "Address": place.get("address", f"{street}, {city}, {state}") if place else f"{street}, {city}, {state}",
#             "Rating": place.get("rating", "N/A") if place else "N/A",
#             "Detailed_User_Reviews": user_reviews
#         }
#         results_list.append(extracted_data)
#         print(f"Collected data: {station_name} ({distance} km away)")

#     except Exception as e:
#         print(f"Error fetching {search_query}")
#     time.sleep(1)

# # ==========================================
# # 6. GENERATE INTERACTIVE MAP (FOLIUM) & FILES
# # ==========================================
# df_results = pd.DataFrame(results_list)

# if not df_results.empty:
#     df_results.to_json('nearby_stations.json', orient='records', indent=4)

#     # Generate Interactive Map
#     m = folium.Map(location=[user_lat, user_lon], zoom_start=13)

#     # 🔵 BLUE PIN: User's Location
#     folium.Marker(
#         [user_lat, user_lon],
#         popup="<b>Your Starting Location</b>",
#         icon=folium.Icon(color="blue", icon="user")
#     ).add_to(m)

#     # 🔴 RED PINS: Top Charging Stations
#     for idx, row in df_results.iterrows():
#         popup_html = f"<b>{row['Station_Name']}</b><br>{row['Distance_km']} km away<br>Rating: {row['Rating']}⭐"
#         folium.Marker(
#             [row['Latitude'], row['Longitude']],
#             popup=popup_html,
#             icon=folium.Icon(color="red", icon="bolt", prefix="fa")
#         ).add_to(m)

#     m.save("interactive_ev_map.html")
#     print("\n✅ Success! Open 'interactive_ev_map.html' in your Colab files panel to view the interactive map with blue and red pins!")

In [ ]:
# # 31-07-2026 (Version 4.0 - Interactive Folium Map with Side Panel & Excel Data)

# !pip install google-search-results pandas openpyxl numpy geopy folium requests

# import pandas as pd
# from serpapi import GoogleSearch
# import time
# import numpy as np
# import requests
# import folium
# import json
# from geopy.geocoders import Nominatim

# # ==========================================
# # 1. HAVERSINE FORMULA (DISTANCE CALCULATION)
# # ==========================================
# def calculate_distance(lat1, lon1, lat2, lon2):
#     R = 6371.0 # Earth radius in kilometers
#     lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

#     dlat = lat2 - lat1
#     dlon = lon2 - lon1

#     a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
#     c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

#     return R * c

# # ==========================================
# # 2. CONFIGURATION VARIABLES
# # ==========================================
# API_KEY = '4d175cb9ae871aa6a0c485c3512c0293d51453c4c26a4fe6f8dd1393ed92fb34'
# EXCEL_PATH = 'https://raw.githubusercontent.com/nov-cpu/8730-project/dashboard-ui/Data/alt_fuel_stations_clean.csv'
# SEARCH_RADIUS_KM = 5.0
# API_LIMIT = 5

# geolocator = Nominatim(user_agent="ev_charging_app_8730")

# # ==========================================
# # 3. GET DYNAMIC USER LOCATION (LIVE OR CUSTOM)
# # ==========================================
# user_lat, user_lon, user_country = None, None, None

# while True:
#     print("\n🌍 EV CHARGING STATION FINDER")
#     print("1. Use Current Live Location (IP-based)")
#     print("2. Enter a Custom Location")
#     choice = input("Select an option (1 or 2): ").strip()

#     if choice == '1':
#         print("\nFetching live location...")
#         try:
#             res = requests.get('http://ip-api.com/json/').json()
#             if res['status'] == 'success':
#                 user_lat, user_lon, user_country = res['lat'], res['lon'], res['country']
#         except Exception as e:
#             print("Could not fetch live location. Please use option 2.")
#             continue

#     elif choice == '2':
#         address = input("\nEnter a starting location (e.g., 'Windsor, ON, Canada'): ")
#         location = geolocator.geocode(address, exactly_one=True)
#         if location:
#             user_lat, user_lon = location.latitude, location.longitude
#             user_country = location.address.split(',')[-1].strip()
#         else:
#             print("Location not found. Please try again.")
#             continue
#     else:
#         print("Invalid choice. Please enter 1 or 2.")
#         continue

#     # --- CONDITION 2.3: Check if location is outside Canada ---
#     if 'Canada' not in user_country and 'CA' not in user_country:
#         print(f"\n❌ ERROR: Your location is detected as '{user_country}'.")
#         print("This application only supports locations within Canada.")
#         print("Please re-enter a valid Canadian location.\n")
#         continue
#     else:
#         print(f"\n✅ Location accepted! Searching near: {user_lat}, {user_lon} (Canada)")
#         break

# # ==========================================
# # 4. LOAD AND FILTER DATA BY PROXIMITY
# # ==========================================
# # Read the single CSV file directly from GitHub
# df_merged = pd.read_csv(EXCEL_PATH)

# df_locations.columns = df_locations.columns.str.strip()
# df_stations.columns = df_stations.columns.str.strip()

# df_merged = pd.merge(df_stations, df_locations, left_on='Location_ID (FK)', right_on='Location_ID (PK)')

# df_merged['Distance_KM'] = calculate_distance(user_lat, user_lon, df_merged['Latitude'], df_merged['Longitude'])
# df_filtered = df_merged[df_merged['Distance_KM'] <= SEARCH_RADIUS_KM].copy()

# total_nearby_count = len(df_filtered)

# # --- CONDITION 2.2: Handle NO nearby stations scenario ---
# if total_nearby_count == 0:
#     min_dist = df_merged['Distance_KM'].min()
#     print(f"\n⚠️ Output: Sorry, there is no charging point option closer (< {SEARCH_RADIUS_KM}kms) to your current location.")
#     print(f"However, if you travel {min_dist:.2f} kms, you can reach the closest available charging station!")

#     df_top_closest = df_merged.sort_values(by="Distance_KM").head(1)
#     scrape_limit = 1
# else:
#     # --- CONDITION 2.1: Handle normal scenario with nearby stations ---
#     print(f"\n--- Output 1: Proximity Count ---")
#     print(f"There are {total_nearby_count} charging locations within {SEARCH_RADIUS_KM}km of your location.")
#     print(f"---------------------------------\n")

#     df_sorted = df_filtered.sort_values(by="Distance_KM")
#     df_top_closest = df_sorted.head(API_LIMIT)
#     scrape_limit = API_LIMIT

# print(f"--- Output 2: Scraping Details & Reviews for Top {scrape_limit} Station(s) ---")

# # ==========================================
# # 5. SCRAPE MAPS & PULL EXCEL METADATA
# # ==========================================
# results_list = []

# for index, row in df_top_closest.iterrows():
#     station_name = row['Station_name']
#     street = row['street_Address']
#     city = row['City']
#     state = row['State']
#     lat = row['Latitude']
#     lon = row['Longitude']
#     distance = round(row['Distance_KM'], 2)

#     # Pulling unique data from Excel
#     ev_pricing = str(row['EV_Pricing']).strip()
#     if ev_pricing == 'nan': ev_pricing = "Pricing unavailable"

#     access_time = str(row['Access_Day_time']).strip()
#     if access_time == 'nan': access_time = "Hours unavailable"

#     search_query = f"EV Charging Station {street} {city} {state}"
#     params = {"engine": "google_maps", "q": search_query, "ll": f"@{lat},{lon},18z", "type": "search", "api_key": API_KEY}

#     try:
#         search = GoogleSearch(params)
#         results = search.get_dict()
#         place = results.get("place_results", results.get("local_results", [None])[0] if "local_results" in results else None)

#         extracted_data = {
#             "Station_Name": station_name,
#             "Distance_km": distance,
#             "Latitude": lat,
#             "Longitude": lon,
#             "Address": place.get("address", f"{street}, {city}, {state}") if place else f"{street}, {city}, {state}",
#             "Rating": place.get("rating", "No Rating") if place else "No Rating",
#             "Reviews_Count": place.get("reviews", 0) if place else 0,
#             "EV_Pricing": ev_pricing,
#             "Access_Time": access_time,
#             "Google_Maps_URL": f"https://www.google.com/maps?q={lat},{lon}"
#         }
#         results_list.append(extracted_data)
#         print(f"Collected data: {station_name} ({distance} km away)")

#     except Exception as e:
#         print(f"Error fetching {search_query}: {e}")
#     time.sleep(1)

# # ==========================================
# # 6. GENERATE INTERACTIVE MAP (FOLIUM) & FILES
# # ==========================================
# df_results = pd.DataFrame(results_list)

# if not df_results.empty:
#     # Save the JSON file for your application
#     df_results.to_json('nearby_stations.json', orient='records', indent=4)

#     # Convert data to JSON string for JavaScript injection
#     stations_json = json.dumps(results_list)

#     # Base Folium Map
#     m = folium.Map(location=[user_lat, user_lon], zoom_start=14)

#     # 🔵 BLUE PIN: User's Location (with simple popup)
#     folium.Marker(
#         [user_lat, user_lon],
#         popup="<b>Your Location</b>",
#         icon=folium.Icon(color="blue", icon="user")
#     ).add_to(m)

#     # 🔴 RED PINS: Stations (Hover tooltip + Click trigger)
#     for idx, row in df_results.iterrows():
#         # HTML tag that triggers the JavaScript function when clicked
#         click_trigger = f'<div style="cursor:pointer; padding:5px;" onclick="openSidePanel({idx})"><b>Click to view details</b></div>'

#         folium.Marker(
#             [row['Latitude'], row['Longitude']],
#             tooltip=f"{row['Station_Name']} ({row['Distance_km']} km)", # Hover text
#             popup=folium.Popup(click_trigger, max_width=150), # Click popup
#             icon=folium.Icon(color="red", icon="bolt", prefix="fa")
#         ).add_to(m)

#     # Get HTML string of map
#     map_html = m.get_root().render()

#     # --- INJECT CUSTOM CSS AND JAVASCRIPT FOR SIDE PANEL ---
#     custom_ui = f"""
#     <style>
#         /* Side Panel Styling */
#         #sidePanel {{
#             height: 100%;
#             width: 0; /* Hidden by default */
#             position: fixed;
#             z-index: 9999;
#             top: 0;
#             left: 0;
#             background-color: #ffffff;
#             overflow-x: hidden;
#             transition: 0.3s;
#             box-shadow: 2px 0 10px rgba(0,0,0,0.3);
#             font-family: Arial, sans-serif;
#         }}
#         .panel-content {{
#             padding: 20px;
#         }}
#         .closebtn {{
#             position: absolute;
#             top: 10px;
#             right: 20px;
#             font-size: 30px;
#             text-decoration: none;
#             color: #888;
#         }}
#         .closebtn:hover {{ color: #000; }}
#         h2 {{ color: #202124; font-size: 22px; margin-bottom: 5px; }}
#         p {{ color: #5f6368; font-size: 14px; margin: 8px 0; }}
#         .rating {{ font-weight: bold; color: #f39c12; }}
#         .section-title {{ font-weight: bold; color: #202124; margin-top: 15px; border-bottom: 1px solid #ddd; padding-bottom: 5px;}}
#         .action-btn {{
#             display: inline-block;
#             margin-top: 20px;
#             padding: 10px 15px;
#             background-color: #1a73e8;
#             color: white;
#             text-decoration: none;
#             border-radius: 20px;
#             font-weight: bold;
#         }}
#         .action-btn:hover {{ background-color: #1557b0; color: white; text-decoration: none;}}
#     </style>

#     <div id="sidePanel">
#         <a href="javascript:void(0)" class="closebtn" onclick="closeSidePanel()">&times;</a>
#         <div class="panel-content" id="panelData">
#             <!-- Data will be injected here by JS -->
#         </div>
#     </div>

#     <script>
#         // Load Python data into JS
#         var stationsData = {stations_json};

#         function openSidePanel(index) {{
#             var station = stationsData[index];
#             var html = `
#                 <h2>${{station.Station_Name}}</h2>
#                 <p class="rating">${{station.Rating}} ⭐ (${{station.Reviews_Count}} reviews)</p>
#                 <p>📍 ${{station.Address}}</p>
#                 <p>📏 ${{station.Distance_km}} km away</p>

#                 <div class="section-title">Pricing</div>
#                 <p>💲 ${{station.EV_Pricing}}</p>

#                 <div class="section-title">Access & Hours</div>
#                 <p>🕒 ${{station.Access_Time}}</p>

#                 <a href="${{station.Google_Maps_URL}}" target="_blank" class="action-btn">Get Directions / View on Maps</a>
#             `;
#             document.getElementById("panelData").innerHTML = html;
#             document.getElementById("sidePanel").style.width = "350px"; // Slide out
#         }}

#         function closeSidePanel() {{
#             document.getElementById("sidePanel").style.width = "0"; // Slide in
#         }}
#     </script>
#     """

#     # Inject UI right before the closing body tag of Folium's HTML
#     final_html = map_html.replace('</body>', f'{custom_ui}</body>')

#     with open("interactive_ev_map.html", "w", encoding='utf-8') as f:
#         f.write(final_html)

#     print("\n✅ Success! Both 'nearby_stations.json' and 'interactive_ev_map.html' are saved.")
#     print("Download the HTML file and open it in your browser to test the interactive Side Panel!")

In [1]:
# 03-08-2026 (Version 5.0)
# ==========================================
# 1. INSTALL REQUIRED PACKAGES
# ==========================================
!pip install google-search-results geopy pandas openpyxl requests

import pandas as pd
import numpy as np
import requests
import json
from serpapi import GoogleSearch
from geopy.geocoders import Nominatim
import time

# ==========================================
# 2. CONFIGURATION & RAW GITHUB LINK
# ==========================================
EXCEL_URL = 'https://raw.githubusercontent.com/nov-cpu/8730-project/Data/Final_Cleaned_Data_(All%20sheets).xlsx'
API_KEY = '4d175cb9ae871aa6a0c485c3512c0293d51453c4c26a4fe6f8dd1393ed92fb34'
SEARCH_RADIUS_KM = 5.0
API_LIMIT = 5

geolocator = Nominatim(user_agent="uwindsor_ev_analytics_8730")

# ==========================================
# 3. RELATIONAL DATABASE JOINS (ETL PHASE)
# ==========================================

import sqlite3

print("Extracting Relational Tables from GitHub...")
excel_sheets = pd.read_excel(EXCEL_URL, sheet_name=None)

df_stations = excel_sheets.get('Stations (Parent Table)')
df_locations = excel_sheets.get('Locations (Entity Table)')

if df_stations is None or df_locations is None:
    print("❌ Error: Could not find the required sheet names. Check your Excel file.")
else:
    df_stations.columns = df_stations.columns.str.strip()
    df_locations.columns = df_locations.columns.str.strip()

    print("Initializing local SQL Database...")
    # Create an in-memory SQL database (or a local .db file)
    conn = sqlite3.connect('ev_infrastructure.db')

    # Load DataFrames into SQL tables
    df_stations.to_sql('stations', conn, index=False, if_exists='replace')
    df_locations.to_sql('locations', conn, index=False, if_exists='replace')

    # Execute a real SQL Query to join the parent and entity tables
    sql_query = """
        SELECT
            s.*,
            l.Latitude,
            l.Longitude,
            l.Street_Address,
            l.City
        FROM stations s
        INNER JOIN locations l
            ON s.[Location_ID (FK)] = l.[Location_ID (PK)]
    """

    # Read the SQL query results back into our pipeline
    df_merged = pd.read_sql_query(sql_query, conn)
    print(f"✅ SQL Pipeline Complete: {len(df_merged)} total records successfully joined via SQL.")

    # Save the merged SQL data as a flat CSV file for the React Dashboard
    df_merged.to_csv('final_merged_dashboard_data.csv', index=False)
    print("CSV file generated! Download it from the folder icon on the left.")





#print("Extracting and Joining Relational Tables from GitHub...")
#excel_sheets = pd.read_excel(EXCEL_URL, sheet_name=None)

#df_stations = excel_sheets.get('Stations (Parent Table)')
#df_locations = excel_sheets.get('Locations (Entity Table)')

#if df_stations is None or df_locations is None:
    #print("❌ Error: Could not find the required sheet names. Check your Excel file.")
#else:
    #df_stations.columns = df_stations.columns.str.strip()
    #df_locations.columns = df_locations.columns.str.strip()

    # Simulating SQL INNER JOIN on Location_ID
    #df_merged = pd.merge(df_stations, df_locations, left_on='Location_ID (FK)', right_on='Location_ID (PK)')
    #print(f"✅ Data Pipeline Complete: {len(df_merged)} total records successfully joined.")

    # Save the merged Excel data as a flat CSV file
    #df_merged.to_csv('final_merged_dashboard_data.csv', index=False)
    #print("CSV file generated! Download it from the folder icon on the left.")

# ==========================================
# 4. HAVERSINE FORMULA (DISTANCE)
# ==========================================
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2)**2
    return R * (2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a)))

# ==========================================
# 5. USER INPUT & PROXIMITY SEARCH
# ==========================================
print("\n🌍 EV CHARGING STATION FINDER")
address = input("Enter a Canadian search location (e.g., 'Windsor, ON'): ")
location = geolocator.geocode(address)

if location:
    user_lat, user_lon = location.latitude, location.longitude
    print(f"\nTarget Acquired: {user_lat:.4f}, {user_lon:.4f}")

    df_merged['Distance_KM'] = calculate_distance(user_lat, user_lon, df_merged['Latitude'], df_merged['Longitude'])
    df_nearby = df_merged[df_merged['Distance_KM'] <= SEARCH_RADIUS_KM].sort_values(by='Distance_KM')

    if df_nearby.empty:
        print(f"No stations found within {SEARCH_RADIUS_KM}km.")
    else:
        df_top = df_nearby.head(API_LIMIT)
        print(f"\nFound {len(df_nearby)} stations in radius. Initiating SerpAPI extraction for closest {API_LIMIT}...\n")

        # ==========================================
        # 6. DATA ENRICHMENT (SERPAPI SCRAPING)
        # ==========================================
        for _, row in df_top.iterrows():
            station_name = row.get('Station_name', 'Unknown')
            city = row.get('City', '')
            veh_class = row.get('Maximum_Vehicle_Class', 'Standard Duty')

            params = {
                "engine": "google_maps",
                "q": f"EV Charging Station {station_name} {city}",
                "ll": f"@{row['Latitude']},{row['Longitude']},16z",
                "api_key": API_KEY
            }

            try:
                search = GoogleSearch(params)
                data = search.get_dict().get("place_results", {})
                rating = data.get("rating", "N/A")
                reviews = data.get("reviews", "N/A")

                print(f"✅ Scraped: {station_name} | Dist: {row['Distance_KM']:.1f}km | Google Rating: {rating} ({reviews} reviews) | Veh Class: {veh_class}")
            except Exception as e:
                print(f"❌ API Error for {station_name}")
            time.sleep(1)
else:
    print("Location not found.")

  Preparing metadata (setup.py) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=96a47e2649fcd34b07509b9fe86e80a4234a1ca09fa7ff8a02b8ee590338fecf
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results
Extracting Relational Tables from GitHub...
Initializing local SQL Database...
✅ SQL Pipeline Complete: 15565 total records successfully joined via SQL.
CSV file generated! Download it from the folder icon on the left.

🌍 EV CHARGING STATION FINDER
Enter a Canadian search location (e.g., 'Windsor, ON'): Toronto On

Target Acquired: 43.6535, -79.3839

Found 514 stations in radius. Initiating SerpAPI extraction for closest 5...

✅ Scraped: COB RBCC 10 | Dist: 0.0km | Google Rating: N/A (N/A reviews) | Veh Class: None
✅ Scraped: 2800 14TH AVE | Dist: 0.1km | Google Rating: N/A (N/A reviews) | Veh Class: None
✅ Scraped: Explora